# CurateAI — AI Commerce Agent
## 5-Moment Shopping Journey Walkthrough

> A narrated walkthrough of CurateAI's complete shopping flow — from Instagram DM to order confirmation.
> Built with Streamlit + Claude claude-opus-4-5 (Anthropic)

**Assignment:** AI-Powered Instagram Commerce Agent Prototype  
**Built by:** Amarjyot Singh (@singh_amarjyot)  
**GitHub:** https://github.com/SINGHAMARJYOT/curateai-shopdm

---
## Setup

In [ ]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY', ''))

PERSONA = "You are CurateAI, a warm personal shopping concierge inside Instagram DM. Be warm, concise (1-2 sentences), natural. Use first name."

def ask_curate(prompt, fallback):
    try:
        resp = client.messages.create(
            model='claude-opus-4-5',
            max_tokens=200,
            messages=[{"role": "user", "content": f"{PERSONA}\n\n{prompt}"}]
        )
        return resp.content[0].text.strip()
    except Exception:
        return fallback

print('✅ Setup complete')

---
## Mock Data — Shopper Profile & Catalog

In [ ]:
SHOPPER = {
    "handle": "@singh_amarjyot",
    "name": "Amarjyot Singh",
    "first_name": "Amarjyot",
    "tier": "Platinum",
    "points": 8240,
    "cashback_rate": 12,
    "purchase_history": ["Roadster", "H&M", "boAt"],
    "style_profile": "Streetwear, casual fashion, tech gadgets, premium audio",
    "addresses": ["Home — Bengaluru, Karnataka"],
    "payment": ["UPI — GPay", "Credit Card — HDFC Millennia"],
}

CATALOG = {
    "F001": {"name":"Oversized Drop Shoulder Hoodie","variant":"Olive","brand":"H&M","mrp":2999,"best_price":1979,"platform":"H&M.com","discount":34,"coupon":"HM30","cashback":180},
    "G001": {"name":"Airdopes 141","variant":"True Wireless Earbuds","brand":"boAt","mrp":1999,"best_price":999,"platform":"Amazon","discount":50,"coupon":"BOAT50","cashback":100},
    "F003": {"name":"Acid Wash Tee","variant":"Black","brand":"Bonkers Corner","mrp":1299,"best_price":909,"platform":"BonkersCorner.com","discount":30,"coupon":"BONK20","cashback":70},
    "F005": {"name":"Oversized Graphic Tee","variant":"White","brand":"Comet","mrp":1499,"best_price":1049,"platform":"Comet.in","discount":30,"coupon":None,"cashback":80},
}

print(f"👤 Shopper: {SHOPPER['name']} | {SHOPPER['tier']} | {SHOPPER['points']:,} LR")
print(f"🛍️  Catalog: {len(CATALOG)} products loaded")

---
## 🔗 Moment 1 — Login & Identity

**What happens:** CurateAI auto-activates when a product photo or link appears in DM. User verifies mobile via OTP (any 6 digits in demo). Profile, address and payment loaded silently from KwikAI network.

**LLM used:** ❌ No — login is pure Python, no API call needed.

In [ ]:
print("=" * 50)
print("MOMENT 1 — LOGIN & IDENTITY")
print("=" * 50)
print()
print("[DM from Diljit Dosanjh]")
print("  Bro check this H&M jacket 🧥")
print()
print("[CurateAI Banner]")
print("  🔗 Product link detected — checking merchant network...")
print()
print("[Login Card]")
print("  👋 Hey! I'm CurateAI — your personal shopping concierge.")
print("  Login once to unlock best prices, loyalty rewards & instant checkout.")
print("  🔒 Only you can see this conversation.")
print()
print("[Mobile Auto-detected]")
print(f"  +91 7014363367 ✓")
print()
print("[OTP Verified]")
print(f"  ✅ Account created!")
print(f"  Profile loaded: {SHOPPER['name']} | {SHOPPER['tier']} | {SHOPPER['points']:,} LR")
print(f"  Address: {SHOPPER['addresses'][0]}")
print(f"  Payment: {SHOPPER['payment'][0]}")

---
## 🛍️ Moment 2 — Shop the Look

**What happens:** CurateAI analyses the product shared in DM. Two paths:
- **Exact match** → Single product card shown
- **Not in network** → Carousel of 3-5 closest alternatives

**LLM used:** ✅ Yes — warm intro message generated by Claude

In [ ]:
print("=" * 50)
print("MOMENT 2 — SHOP THE LOOK")
print("=" * 50)
print()

# Demo: Diljit sends H&M jacket → exact match → F001
pid = "F001"
p = CATALOG[pid]

prompt = f"Found exact product: {p['name']} by {p['brand']} at ₹{p['best_price']} on {p['platform']}. Tell {SHOPPER['first_name']} in 1 warm sentence."
fallback = f"Found it, {SHOPPER['first_name']}! Here's the {p['name']} by {p['brand']} — exactly what was shared, at ₹{p['best_price']}."

msg = ask_curate(prompt, fallback)

print("[Trigger] Diljit Dosanjh → Streetwear jacket link → EXACT MATCH")
print()
print(f"✦ CurateAI: {msg}")
print()
print("[Product Card]")
print(f"  {p['brand']} — {p['name']} ({p['variant']})")
print(f"  ₹{p['best_price']:,}  ~~₹{p['mrp']:,}~~  {p['discount']}% OFF")
print(f"  via {p['platform']} · KwikAI")
print(f"  🏷️  Coupon: {p['coupon']}")
print()
print("[Feedback] Is this the same product?")
print("  → Yes — Get Best Price  |  → Not quite")

In [ ]:
# Demo: Sonam Bajwa sends ethnic wear → NOT in network → carousel
print("[Trigger] Sonam Bajwa → Fashion photo → NOT IN NETWORK → Carousel")
print()

similars = ["F001", "F003", "F005"]
prompt2 = f"Showing {len(similars)} product recommendations for {SHOPPER['first_name']} who loves {SHOPPER['style_profile']}. 1 warm sentence."
fallback2 = f"These picks match your style perfectly, {SHOPPER['first_name']} — sorted by best discount available."

msg2 = ask_curate(prompt2, fallback2)
print(f"✦ CurateAI: {msg2}")
print()
print("[Carousel — You May Also Like]")
for pid in similars:
    p = CATALOG[pid]
    print(f"  {p['brand']} · {p['name']} · ₹{p['best_price']:,} ({p['discount']}% OFF)")

---
## 💰 Moment 3 — Best Price

**What happens:** Agent scans merchant network, surfaces lowest price, flags active coupons and cashback.

**LLM used:** ✅ Yes — price narrative generated by Claude

In [ ]:
print("=" * 50)
print("MOMENT 3 — BEST PRICE")
print("=" * 50)
print()

pid = "F001"
p = CATALOG[pid]

prompt = f"Best price for {p['name']}: ₹{p['best_price']} on {p['platform']} ({p['discount']}% off ₹{p['mrp']}). Cashback ₹{p['cashback']} LR. Coupon: {p['coupon']}. 1-2 sentences."
fallback = f"Scanned the network — best deal is ₹{p['best_price']} on {p['platform']} ({p['discount']}% off MRP). You'll also earn ₹{p['cashback']} LR cashback."

msg = ask_curate(prompt, fallback)
print(f"✦ CurateAI: {msg}")
print()
print("[Price Breakdown Card]")
print(f"  MRP:                  ₹{p['mrp']:,}")
print(f"  Merchant discount:   −₹{p['mrp']-p['best_price']:,} ({p['discount']}%)")
print(f"  Cashback earned:     +₹{p['cashback']:,} LR")
print(f"  ─────────────────────────")
print(f"  Best Price:           ₹{p['best_price']:,}")
print(f"  via {p['platform']} · KwikAI")
if p['coupon']:
    print(f"  🏷️  Apply code: {p['coupon']}")

---
## 🎁 Moment 4 — Loyalty Apply

**What happens:** Agent surfaces shopper's reward balance and applies points to the order. Calculation based on cart value tier.

**LLM used:** ✅ Yes — celebratory loyalty message generated by Claude

In [ ]:
print("=" * 50)
print("MOMENT 4 — LOYALTY APPLY")
print("=" * 50)
print()

def calc_loyalty(cart_value, points):
    if cart_value <= 1000:      r_pct, lr_rate, e_pct, e_rate = 0.10, 0.5,  0.075, 0.2
    elif cart_value <= 2000:    r_pct, lr_rate, e_pct, e_rate = 0.075, 0.35, 0.05,  0.35
    elif cart_value <= 5000:    r_pct, lr_rate, e_pct, e_rate = 0.05,  0.30, 0.035, 0.4
    else:                       r_pct, lr_rate, e_pct, e_rate = 0.035, 0.25, 0.025, 0.5
    redeem_pts = min(points, cart_value * r_pct)
    return {
        "redeemable_pts": int(redeem_pts),
        "discount_rs": round(redeem_pts * lr_rate, 2),
        "points_earned": int(cart_value * e_pct * e_rate),
        "lr_rate": lr_rate,
    }

pid = "F001"
p = CATALOG[pid]
ld = calc_loyalty(p["best_price"], SHOPPER["points"])

prompt = f"{SHOPPER['first_name']} has {SHOPPER['points']:,} LR ({SHOPPER['tier']}). Redeeming {ld['redeemable_pts']:,} pts = ₹{ld['discount_rs']:.0f} off. Earning {ld['points_earned']} pts. 1-2 celebratory sentences."
fallback = f"Your {SHOPPER['tier']} rewards are working for you, {SHOPPER['first_name']}! Redeeming {ld['redeemable_pts']:,} points saves you ₹{ld['discount_rs']:.0f} — applied instantly."

msg = ask_curate(prompt, fallback)
print(f"✦ CurateAI: {msg}")
print()
print("[Loyalty Card]")
print(f"  ★ {SHOPPER['tier']} Member")
print(f"  Available: {SHOPPER['points']:,} LR")
print(f"  Redeemable on this order: {ld['redeemable_pts']:,} pts")
print(f"  Discount applied:         ₹{ld['discount_rs']:,.0f}")
print(f"  Rate:                     1 LR = ₹{ld['lr_rate']}")
print(f"  Points you'll earn:       +{ld['points_earned']:,} LR")

---
## ✅ Moment 5 — Checkout & Order Success

**What happens:** Pre-filled cart with identity, address, payment, rewards all applied. One tap to checkout. Order confirmation sent back in DM.

**LLM used:** ✅ Yes — checkout handover + order success message generated by Claude

In [ ]:
import random

print("=" * 50)
print("MOMENT 5 — CHECKOUT & ORDER SUCCESS")
print("=" * 50)
print()

pid = "F001"
p = CATALOG[pid]
ld = calc_loyalty(p["best_price"], SHOPPER["points"])
final = p["best_price"] - ld["discount_rs"]
saved = p["mrp"] - final
order_id = random.randint(100000, 999999)

# Checkout message
prompt_co = f"Cart ready for {SHOPPER['first_name']}: {p['name']} at ₹{final:.0f} (saved ₹{saved:.0f}). Identity pre-filled, rewards applied. 1-2 sentences, VIP concierge tone."
fallback_co = f"Your cart is ready, {SHOPPER['first_name']} — ₹{final:.0f} all in, with every deal applied. One tap and it's yours."
msg_co = ask_curate(prompt_co, fallback_co)

print(f"✦ CurateAI: {msg_co}")
print()
print("[Checkout Card]")
print(f"  📍 {SHOPPER['addresses'][0]}")
print(f"  💳 {SHOPPER['payment'][0]}")
print(f"  {p['name']}         ₹{p['mrp']:,}")
print(f"  Merchant deal        −₹{p['mrp']-p['best_price']:,}")
print(f"  Loyalty points       −₹{ld['discount_rs']:,.0f}")
print(f"  Cashback earned      +₹{p['cashback']:,} LR")
print(f"  ─────────────────────────")
print(f"  Total:               ₹{final:,.0f}")
print(f"  You save:            ₹{saved:,.0f} 🎉")
print()

# Order success
prompt_ok = f"Order confirmed for {SHOPPER['first_name']}! {p['name']} on its way. Earned {ld['points_earned']} LR. 1 celebratory sentence."
fallback_ok = f"Order confirmed! Your {p['name']} is on its way, {SHOPPER['first_name']} — and you just earned {ld['points_earned']} LR points. 🎉"
msg_ok = ask_curate(prompt_ok, fallback_ok)

print(f"✦ CurateAI: {msg_ok}")
print()
print("[Order Confirmation]")
print(f"  ✅ Order Confirmed!")
print(f"  Order ID:       KWK{order_id}")
print(f"  Product:        {p['name']}")
print(f"  Delivering to:  Home, Bengaluru")
print(f"  Est. delivery:  Apr 9–11, 2026")
print(f"  Points earned:  +{ld['points_earned']:,} LR")

---
## 📊 Journey Summary

| Moment | Screen | LLM Used | Key Output |
|--------|--------|----------|------------|
| 1 | Login & Identity | ❌ No | OTP verified, profile loaded |
| 2 | Shop the Look | ✅ Yes | Exact product card or carousel |
| 3 | Best Price | ✅ Yes | Price breakdown with coupon |
| 4 | Loyalty Apply | ✅ Yes | Points redeemed, savings shown |
| 5 | Checkout + Success | ✅ Yes | Pre-filled cart + order confirmed |

---

**Total LLM calls per journey:** 4–5 (one per moment)  
**Fallback coverage:** 100% — app runs without API key  
**Model:** `claude-opus-4-5` (Anthropic)  
**Live app:** https://curateai-shopdm.streamlit.app